# CP5 - Prompt & IA — Banco Multiagente PIX

Evolução do sistema construído na CP4: agora os agentes leem e alteram os dados reais dos arquivos `saldo.csv`, `contatos_pix.csv` e `transacoes_pix.csv`, em vez de usar dados fictícios fixos em memória.

**Novas funcionalidades do Agente PIX:**
- Consultar saldo (agora a partir de `saldo.csv`)
- Listar contatos
- Adicionar contatos
- Buscar contatos
- Realizar PIX (mantido da CP4, agora persistindo em `saldo.csv` e `transacoes_pix.csv`)


### Instalação e configuração

In [ ]:
%pip install -q -U openai-agents pandas

In [ ]:
import pandas as pd
from agents import Agent, Runner, function_tool

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Adicione OPENAI_API_KEY aos Secrets do Google Colab.")

os.environ["OPENAI_API_KEY"] = api_key
print("Chave configurada com segurança.")

### Localização dos arquivos de dados

Os arquivos `saldo.csv`, `contatos_pix.csv` e `transacoes_pix.csv` já devem estar na raiz do Colab. Não é necessário fazer upload manual.

In [ ]:
ARQ_SALDO = "saldo.csv"
ARQ_CONTATOS = "contatos_pix.csv"
ARQ_TRANSACOES = "transacoes_pix.csv"

### Carregamento dos dados em DataFrames

In [ ]:
contatos_df = pd.read_csv(ARQ_CONTATOS)
saldo_df = pd.read_csv(ARQ_SALDO)
transacoes_df = pd.read_csv(ARQ_TRANSACOES)

# Cliente logado nesta sessão (único cliente presente em saldo.csv)
CLIENTE_ID = int(saldo_df.iloc[0]["id_cliente"])

print(f"Cliente logado: id {CLIENTE_ID} — {saldo_df.iloc[0]['nome']}")
print(f"Saldo inicial: R$ {saldo_df.iloc[0]['saldo']:,.2f}")
print(f"Contatos cadastrados: {(contatos_df['id_cliente'] == CLIENTE_ID).sum()}")

### Tools do Agente PIX

Cada tool lê e/ou grava diretamente nos arquivos CSV, para que as alterações (novo contato, novo saldo, nova transação) persistam de verdade.

In [ ]:
@function_tool
def consultar_saldo() -> dict:
    """
    Retorna o saldo atual do cliente logado, lendo diretamente de saldo.csv.
    Use sempre que o cliente perguntar sobre o saldo disponível ou antes de um PIX.
    """
    linha = saldo_df.loc[saldo_df["id_cliente"] == CLIENTE_ID]
    if linha.empty:
        return {"erro": f"Cliente {CLIENTE_ID} não encontrado em saldo.csv."}
    linha = linha.iloc[0]
    return {"titular": linha["nome"], "saldo_disponivel": float(linha["saldo"])}


@function_tool
def listar_contatos() -> dict:
    """
    Lista todos os contatos PIX cadastrados pelo cliente logado, lendo contatos_pix.csv.
    Use quando o cliente pedir para ver todos os seus contatos.
    """
    contatos = contatos_df.loc[contatos_df["id_cliente"] == CLIENTE_ID]
    if contatos.empty:
        return {"contatos": [], "mensagem": "Nenhum contato cadastrado."}
    return {"contatos": contatos[["nome_contato", "chave_pix"]].to_dict(orient="records")}


@function_tool
def buscar_contato(nome: str) -> dict:
    """
    Busca um contato do cliente logado pelo nome (busca parcial, sem diferenciar
    maiúsculas/minúsculas), lendo contatos_pix.csv.

    Args:
        nome: Nome (ou parte do nome) do contato a localizar.
    """
    nome_lower = nome.lower().strip()
    contatos = contatos_df.loc[contatos_df["id_cliente"] == CLIENTE_ID]
    encontrados = contatos[contatos["nome_contato"].str.lower().str.contains(nome_lower)]

    if encontrados.empty:
        return {"erro": f"Nenhum contato encontrado para '{nome}'."}

    if len(encontrados) > 1:
        return {
            "aviso": "Mais de um contato encontrado. Peça ao usuário para confirmar qual deseja.",
            "contatos": encontrados[["nome_contato", "chave_pix"]].to_dict(orient="records"),
        }

    r = encontrados.iloc[0]
    return {"contato": {"nome_contato": r["nome_contato"], "chave_pix": r["chave_pix"]}}


@function_tool
def adicionar_contato(nome_contato: str, chave_pix: str) -> dict:
    """
    Adiciona um novo contato PIX para o cliente logado e salva em contatos_pix.csv.
    Não permite cadastrar dois contatos com o mesmo nome.

    Args:
        nome_contato: Nome do novo contato.
        chave_pix: Chave PIX do novo contato (e-mail, celular, CPF/CNPJ ou aleatória).
    """
    global contatos_df

    ja_existe = contatos_df[
        (contatos_df["id_cliente"] == CLIENTE_ID)
        & (contatos_df["nome_contato"].str.lower() == nome_contato.lower())
    ]
    if not ja_existe.empty:
        return {"erro": f"Já existe um contato chamado '{nome_contato}'."}

    nova_linha = {"id_cliente": CLIENTE_ID, "nome_contato": nome_contato, "chave_pix": chave_pix}
    contatos_df = pd.concat([contatos_df, pd.DataFrame([nova_linha])], ignore_index=True)
    contatos_df.to_csv(ARQ_CONTATOS, index=False)

    return {"sucesso": True, "contato_adicionado": nova_linha}


@function_tool
def realizar_pix(nome_destinatario: str, chave_pix: str, valor: float) -> dict:
    """
    Executa um PIX simulado: debita o valor do saldo do cliente logado (saldo.csv)
    e registra a transação em transacoes_pix.csv.
    Deve ser chamada APENAS após o usuário confirmar os dados da transação.

    Args:
        nome_destinatario: Nome do destinatário.
        chave_pix: Chave PIX do destinatário.
        valor: Valor em reais a transferir (deve ser positivo).
    """
    global saldo_df, transacoes_df
    import uuid
    import datetime

    if valor <= 0:
        return {"erro": "O valor do PIX deve ser maior que zero."}

    linha_saldo = saldo_df.loc[saldo_df["id_cliente"] == CLIENTE_ID]
    saldo_atual = float(linha_saldo.iloc[0]["saldo"])

    if valor > saldo_atual:
        return {"erro": "Saldo insuficiente.", "saldo_disponivel": saldo_atual, "valor_solicitado": valor}

    novo_saldo = saldo_atual - valor
    saldo_df.loc[saldo_df["id_cliente"] == CLIENTE_ID, "saldo"] = novo_saldo
    saldo_df.to_csv(ARQ_SALDO, index=False)

    nova_transacao = {
        "id_transacao": str(uuid.uuid4()),
        "id_cliente": CLIENTE_ID,
        "destinatario": nome_destinatario,
        "chave_pix": chave_pix,
        "valor": valor,
        "data": datetime.datetime.now().isoformat(),
    }
    transacoes_df = pd.concat([transacoes_df, pd.DataFrame([nova_transacao])], ignore_index=True)
    transacoes_df.to_csv(ARQ_TRANSACOES, index=False)

    return {"sucesso": True, "transacao": nova_transacao, "novo_saldo": novo_saldo}

### Definição dos Agentes

Mesma arquitetura da CP4 — três agentes:
- **Agente de Dúvidas** — responde sobre produtos e serviços do banco
- **Agente PIX** — opera a conta com as tools acima (agora baseadas em CSV)
- **Agente de Atendimento** — triagem; faz handoff para o especialista correto

In [ ]:
# Dúvidas
agente_duvidas = Agent(
    name="Agente de Dúvidas",
    handoff_description=(
        "Use para perguntas sobre produtos, serviços, tarifas, limites, "
        "cartões, seguros, investimentos, PIX (funcionamento geral) e "
        "regras do banco. NÃO use para operações de conta, contatos ou "
        "transferências — essas ficam com o Agente PIX."
    ),
    instructions=(
        "Você é o assistente de dúvidas do Banco Fictício S.A. "
        "Responda em português, com clareza e cordialidade. "
        "Use as informações abaixo sobre o banco:\n\n"

        "=== PRODUTOS E SERVIÇOS ===\n"
        "- Conta Corrente: sem tarifa para clientes com renda acima de R$1.500/mês.\n"
        "- Conta Poupança: rendimento de 0,5% a.m. + TR. Sem tarifas.\n"
        "- Cartão de Crédito Básico: limite inicial de R$1.000. Anuidade: R$199/ano (isenta com 1 compra/mês).\n"
        "- Cartão de Crédito Gold: limite inicial de R$5.000. Anuidade: R$49/mês.\n"
        "- Seguro Vida: a partir de R$29/mês. Cobertura de até R$200.000.\n"
        "- CDB: liquidez diária, 100% do CDI. Aplicação mínima: R$500.\n"
        "- Tesouro Direto: disponível pelo app, sem taxa de custódia adicional do banco.\n\n"

        "=== TARIFAS ===\n"
        "- TED: R$8,50 por operação (gratuita via app).\n"
        "- DOC: R$6,00 por operação.\n"
        "- PIX: GRATUITO, 24h por dia, 7 dias por semana.\n"
        "- Saque em caixas do banco: 4 gratuitos/mês; excedentes R$4,50 cada.\n"
        "- Extrato impresso: R$2,00 por via.\n\n"

        "=== PIX — REGRAS ===\n"
        "- Limite diário padrão: R$5.000 (período diurno) e R$1.000 (período noturno, 20h–6h).\n"
        "- Chaves aceitas: CPF, CNPJ, e-mail, celular, chave aleatória.\n"
        "- Prazo de devolução (Pix errado): até 90 dias, mediante solicitação.\n\n"

        "Se não souber a resposta, diga claramente e sugira contato pelo telefone 0800-123-4567."
    ),
    model="gpt-4o-mini",
)

# Pix
agente_pix = Agent(
    name="Agente PIX",
    handoff_description=(
        "Use para operações na conta do cliente: consultar saldo, listar contatos, "
        "buscar contatos, adicionar contatos e enviar PIX. "
        "NÃO use para dúvidas gerais sobre o banco."
    ),
    instructions=(
        "Você é o agente operacional PIX do Banco Fictício S.A. "
        "Seu papel é executar operações de conta e de contatos com segurança e transparência.\n\n"

        "FUNCIONALIDADES DISPONÍVEIS:\n"
        "- consultar_saldo: informar o saldo atual.\n"
        "- listar_contatos: listar todos os contatos PIX cadastrados.\n"
        "- buscar_contato: localizar um contato específico pelo nome.\n"
        "- adicionar_contato: cadastrar um novo contato (peça nome e chave PIX se faltar algum dado).\n"
        "- realizar_pix: executar uma transferência.\n\n"

        "FLUXO OBRIGATÓRIO para um PIX:\n"
        "1. Use 'buscar_contato' para identificar o destinatário pelo nome informado. "
           "Se não existir, ofereça usar 'adicionar_contato' antes de prosseguir.\n"
        "2. Use 'consultar_saldo' para verificar se há saldo suficiente.\n"
        "3. Apresente ao usuário um RESUMO DA TRANSAÇÃO com: destinatário, chave PIX, valor e saldo atual. "
           "Peça confirmação explícita ('Confirma? S/N').\n"
        "4. Somente após confirmação do usuário, chame 'realizar_pix'.\n"
        "5. Informe o comprovante com ID da transação e novo saldo.\n\n"

        "REGRAS:\n"
        "- Nunca execute um PIX sem confirmação explícita do usuário.\n"
        "- Se o saldo for insuficiente, informe o valor disponível.\n"
        "- Responda sempre em português, com cordialidade e objetividade."
    ),
    tools=[consultar_saldo, listar_contatos, buscar_contato, adicionar_contato, realizar_pix],
    model="gpt-4o-mini",
)

# Atendimento
agente_atendimento = Agent(
    name="Agente de Atendimento",
    instructions=(
        "Você é o recepcionista virtual do Banco Fictício S.A. "
        "Cumprimente o cliente com cordialidade e identifique a intenção da solicitação.\n\n"
        "REGRAS DE ROTEAMENTO:\n"
        "- Solicitações de PIX, transferência, saldo ou contatos (listar/buscar/adicionar) "
           "→ handoff para o Agente PIX.\n"
        "- Dúvidas sobre produtos, tarifas, serviços, cartões, investimentos ou regras do banco "
           "→ handoff para o Agente de Dúvidas.\n"
        "- Solicitações ambíguas: peça uma breve clarificação antes de fazer o handoff.\n\n"
        "Seja breve na recepção. O especialista cuidará dos detalhes."
    ),
    handoffs=[agente_pix, agente_duvidas],
    model="gpt-4o-mini",
)

### Função auxiliar de teste

In [ ]:
async def atender(solicitacao: str, historico: list = None) -> tuple[str, list]:
    """
    Executa uma solicitação a partir do Agente de Atendimento.
    Retorna (resposta_final, historico_atualizado).
    """
    msgs = (historico or []) + [{"role": "user", "content": solicitacao}]

    resultado = await Runner.run(agente_atendimento, msgs)

    print(f"Agente que concluiu: {resultado.last_agent.name}")
    print("─" * 60)
    print(resultado.final_output)
    print("─" * 60)

    novo_historico = msgs + [{"role": "assistant", "content": resultado.final_output}]
    return resultado.final_output, novo_historico

## Chat interativo

In [ ]:
historico_chat = []

print("Banco — Atendimento Virtual")
print("Digite 'sair' para encerrar.")
print("=" * 50)

while True:
    mensagem = input("\nVocê: ").strip()
    if not mensagem:
        continue
    if mensagem.lower() in ("sair", "exit", "quit"):
        print("\nVocê saiu")
        break

    print()
    _, historico_chat = await atender(mensagem, historico_chat)